# **EMPLOYEE ATTRITION PREDICTION - LOGISTIC REGRESSION MODEL**
---


## **1. IMPORT LIBRARIES**


In [1]:
import pandas as pd
import numpy as np


In [2]:

# Database
from sqlalchemy import create_engine


In [3]:

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [4]:

# Saving Model
import joblib


In [5]:

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="whitegrid")



---


## **2. LOAD DATA FROM MYSQL**


In [6]:
username = "root"
password = "pri0000"
host = "localhost"
database = "hr_analytics"

try:
    engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")
    df = pd.read_sql("SELECT * FROM employee_attrition", con=engine)
    print(f"Data Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print("Error loading data:", e)
    exit()



Data Loaded Successfully: 1470 rows, 32 columns


---


## **3. DATA CLEANING & FEATURE ENGINEERING**


In [7]:

# 3.1 Create Target Variable
df["attrition_flag"] = df["attrition"].map({"Yes": 1, "No": 0})


In [8]:

# 3.2 Drop Columns Not Useful for Modeling
cols_to_drop = [
    "attrition",
    "employeecount",
    "over18",
    "standardhours",
    "employeenumber"
]
df_ml = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


In [9]:

# 3.3 One-Hot Encoding
df_ml = pd.get_dummies(df_ml, drop_first=True)

print(f"Dataset Ready for Modeling: {df_ml.shape}")



Dataset Ready for Modeling: (1470, 45)


---


## **4. TRAIN-TEST SPLIT**


In [10]:
X = df_ml.drop("attrition_flag", axis=1)
y = df_ml["attrition_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



---


## **5. FEATURE SCALING**


In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



---


## **6. MODEL TRAINING**


In [12]:
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)



---


## **7. MODEL EVALUATION**


In [13]:
accuracy = round(accuracy_score(y_test, y_pred) * 100, 2)
print(f"\nModel Accuracy: {accuracy}%")



Model Accuracy: 86.05%


In [14]:

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Confusion Matrix:
[[237  10]
 [ 31  16]]


In [15]:

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       247
           1       0.62      0.34      0.44        47

    accuracy                           0.86       294
   macro avg       0.75      0.65      0.68       294
weighted avg       0.84      0.86      0.84       294



---



## **8. FEATURE IMPORTANCE**


In [16]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values(by="coefficient", ascending=False)


In [17]:

print("\nTop 10 Factors Increasing Attrition:")
print(coef_df.head(10))



Top 10 Factors Increasing Attrition:
                             feature  coefficient
43                      overtime_Yes     0.864567
23  businesstravel_Travel_Frequently     0.751247
34     jobrole_Laboratory Technician     0.714756
21           yearssincelastpromotion     0.528704
11                numcompaniesworked     0.487609
40      jobrole_Sales Representative     0.481459
24      businesstravel_Travel_Rarely     0.450070
39           jobrole_Sales Executive     0.409361
2                   distancefromhome     0.393594
42              maritalstatus_Single     0.376877


In [18]:

print("\nTop 10 Factors Reducing Attrition:")
print(coef_df.tail(10))




Top 10 Factors Reducing Attrition:
                         feature  coefficient
20            yearsincurrentrole    -0.356016
27  educationfield_Life Sciences    -0.356181
37     jobrole_Research Director    -0.361341
6                 jobinvolvement    -0.365340
22          yearswithcurrmanager    -0.398368
0                            age    -0.407597
29        educationfield_Medical    -0.414529
8                jobsatisfaction    -0.419417
4        environmentsatisfaction    -0.481659
16             totalworkingyears    -0.555603


---


## **9. SAVE MODEL FOR DEPLOYMENT**


In [19]:
joblib.dump(model, "logistic_model_attrition.pkl")
joblib.dump(scaler, "scaler_attrition.pkl")


['scaler_attrition.pkl']

In [20]:

print("\nModel and Scaler saved successfully.")



Model and Scaler saved successfully.


---